In [4]:
import os
import pandas as pd
import numpy as np
from PIL import Image

CSV_PATH = '../data/nacti_metadata.csv'
DATA_DIR = '../data/'
PROCESS_ALL = False
TARGET_SIZE = (224, 224)

def load_labels(csv_path):
    df = pd.read_csv(csv_path, low_memory = False)
    label_dict = dict(zip(df['filename'], df['common_name']))
    return label_dict

def preprocess_image(img_path, target_size):
    try:
        img = Image.open(img_path).convert('RGB')
        img = img.resize(target_size)
        img_array = np.array(img) / 255.0
        return img_array
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return None

def prepare_data():
    # 1. Create the dictionary mapping
    label_dict = load_labels(CSV_PATH)
    
    images_data = []
    labels = []
    
    # 2. Loop through part1 to part4
    for i in range(0, 4):
        part_folder = f"part{i}"
        part_path = os.path.join(DATA_DIR, part_folder)
        
        if not os.path.exists(part_path):
            print(f"Warning: {part_path} does not exist. Skipping.")
            continue
            
        # Get all subfolders within the part directory
        subfolders = [f.path for f in os.scandir(part_path) if f.is_dir()]
        if not subfolders:
            continue
            
        # 3. Handle True/False flag for subfolders
        subfolders_to_process = subfolders if PROCESS_ALL else [subfolders[0]]
        
        for subfolder in subfolders_to_process:
            # Gather valid image files
            image_files = [
                f.path for f in os.scandir(subfolder) 
                if f.is_file() and f.name.lower().endswith(('.jpg', '.jpeg', '.png'))
            ]
            
            # Handle True/False flag for images per subfolder
            images_to_process = image_files if PROCESS_ALL else image_files[:10]
            
            for img_path in images_to_process:
                # 4. Generate the dictionary key.
                rel_path = os.path.relpath(img_path, DATA_DIR).replace('\\', '/')
                
                # Match to dictionary, defaulting to "unlabeled" if not found
                label = label_dict.get(rel_path, "unlabeled")
                
                # 5. Prepare the image for CNN
                img_array = preprocess_image(img_path, TARGET_SIZE)
                
                if img_array is not None:
                    images_data.append(img_array)
                    labels.append(label)
                    
    # Convert resulting lists to numpy arrays to plug straight into a CNN
    X = np.array(images_data)
    y = np.array(labels)
    
    return X, y

if __name__ == "__main__":
    print("Loading and preparing data...")
    X, y = prepare_data()
    
    print("\n--- Data Preparation Complete ---")
    print(f"Total images processed: {len(X)}")
    print(f"Feature array (X) shape: {X.shape}") # Expected: (N, 224, 224, 3)
    print(f"Label array (y) shape: {y.shape}")   # Expected: (N,)
    
    if len(y) > 0:
        print(f"Sample labels: {y[:5]}")

Loading and preparing data...


FileNotFoundError: [Errno 2] No such file or directory: '../data/nacti_metadata.csv'

In [2]:
import os; os.getcwd()

'/sfs/weka/scratch/nvu5gw/Project-3/DS-4002-project-3/scripts'

In [1]:
import pandas as pd

filepath = '/sfs/weka/scratch/nvu5gw/Project-3/DS-4002-project-3/data/nacti_metadata.csv'

df = pd.read_csv(filepath, low_memory = False)

print(f"Total data rows: {len(df)}")

FileNotFoundError: [Errno 2] No such file or directory: '/sfs/weka/scratch/nvu5gw/Project-3/DS-4002-project-3/data/nacti_metadata.csv'